# Descarga BIOMASS (ESA) por AOI — Google Colab

**AOIs ya incluidos:** `BOSQUE_NW_02` y `ESTEPA_NW_02` (Patagonia). No hay que subir archivos.

Descarga cada AOI **por separado** (carpetas distintas), convirtiendo solo tu token offline.

### Pasos
1. Corré las celdas en orden con el botón ▶.
2. En la celda del token, pegá tu token offline (de *Copy access token*).
3. La última celda descarga, separado por AOI.

> **Tipos de producto:** `SCS` = imagen compleja de radar. `S1/S2/S3` = las 3 franjas.
> **`1S`** = Standard (TRAE la imagen — usá estos). **`1M`** = Monitoring (liviano, SIN imagen).


## 1) Instalar librerías


In [ ]:
!pip -q install pystac-client requests shapely
print('Librerias instaladas OK')

## 2) (Recomendado) Montar Google Drive
Para que las descargas queden guardadas (Colab borra `/content` al cerrar). Te pide permiso: aceptá.
Si preferís no usar Drive, poné `USAR_DRIVE = False`.


In [ ]:
import os
USAR_DRIVE = True
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SALIDA = '/content/drive/MyDrive/BIOMASS/03_Images'
else:
    SALIDA = '/content/BIOMASS/03_Images'
os.makedirs(SALIDA, exist_ok=True)
print('Salida:', SALIDA)

## 3) Pegá tu token offline (se convierte solo a access token)


In [ ]:
import requests
OFFLINE_TOKEN = os.environ.get('BIOMASS_TOKEN','')   # NO pegar el token aca

TOKEN_URL='https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token'
def obtener_access_token():
    if not OFFLINE_TOKEN.strip(): return ''
    r=requests.post(TOKEN_URL,data={'client_id':'offline-token',
        'client_secret':os.environ.get('BIOMASS_CLIENT_SECRET',''),
        'grant_type':'refresh_token','refresh_token':OFFLINE_TOKEN.strip(),
        'scope':'offline_access openid'},timeout=60)
    r.raise_for_status(); return r.json()['access_token']
ACCESS_TOKEN = obtener_access_token()
print('Access token OK.' if ACCESS_TOKEN else 'Sin token: solo BUSQUEDA (sin descarga).')

## 4) AOIs, configuración y funciones
Fechas amplias para captar toda la cobertura (ESTEPA tuvo datos en verano; BOSQUE en otoño).


In [ ]:
from pystac_client import Client
from shapely.geometry import Polygon

AOIS = {
  'BOSQUE_NW_02': [(-71.5977324,-42.6912349),(-71.4147861,-42.6952402),
                   (-71.4095719,-42.5602750),(-71.5921243,-42.5562884),(-71.5977324,-42.6912349)],
  'ESTEPA_NW_02': [(-71.2137477,-42.8433829),(-71.0303041,-42.8467842),
                   (-71.0258965,-42.7117874),(-71.2089427,-42.7084021),(-71.2137477,-42.8433829)],
}
CATALOG_URL='https://catalog.maap.eo.esa.int/catalogue/'
FECHA=['2025-09-01T00:00:00Z','2026-12-31T23:59:59Z']
COLS=['BiomassLevel1a','BiomassLevel1b','BiomassLevel1c','BiomassLevel2a']
SOLO_1S = True          # True = solo productos con imagen (1S)
MAX_POR_AOI = None      # None = todos; o un numero para probar (ej. 1)

def es_1S(it): return ('__1S' in it.id) or ('_1S_' in it.id)

def descargar_item(item, carpeta):
    global ACCESS_TOKEN
    os.makedirs(carpeta, exist_ok=True)
    for na, asset in item.assets.items():
        dest=os.path.join(carpeta, f'{item.id}__{na}')
        for i in (1,2):
            try:
                h={'Authorization': f'Bearer {ACCESS_TOKEN}'}
                with requests.get(asset.href, headers=h, stream=True, timeout=600) as r:
                    if r.status_code==401 and i==1: ACCESS_TOKEN=obtener_access_token(); continue
                    r.raise_for_status()
                    with open(dest,'wb') as f:
                        for c in r.iter_content(1024*1024): f.write(c)
                print('      OK ->', os.path.basename(dest)); break
            except Exception as e:
                if i==2: print('      ERROR', na, ':', e)
print('AOIs:', list(AOIS.keys()), '| Fechas:', FECHA[0][:10], 'a', FECHA[1][:10])

## 5) Buscar y descargar — SEPARADO POR AOI
Guarda en `03_Images/BOSQUE_NW_02/...` y `03_Images/ESTEPA_NW_02/...`.


In [ ]:
cat=Client.open(CATALOG_URL)
print('Conectado al catalogo STAC de ESA MAAP\n')
resumen={}
for nombre,pts in AOIS.items():
    poly=Polygon(pts); bbox=list(poly.bounds)
    print(f'=== {nombre}  bbox={[round(v,4) for v in bbox]} ===')
    resumen[nombre]={}
    for col in COLS:
        try:
            items=list(cat.search(collections=[col],bbox=bbox,datetime=FECHA,max_items=300).items())
            if SOLO_1S: items=[it for it in items if es_1S(it)]
            resumen[nombre][col]=len(items); print(f'   {col}: {len(items)} producto(s)')
            if ACCESS_TOKEN:
                sel = items if MAX_POR_AOI is None else items[:MAX_POR_AOI]
                for it in sel:
                    print('     Descargando', it.id, '...')
                    descargar_item(it, os.path.join(SALIDA, nombre, col))
        except Exception as e:
            print(f'   {col}: ERROR {e}')
    print()
print('LISTO. Archivos en', SALIDA)
print(resumen)

## 6) (Opcional) Comprimir cada AOI en un .zip y bajarlo a tu PC


In [ ]:
import shutil
from google.colab import files
for nombre in AOIS:
    carpeta=os.path.join(SALIDA, nombre)
    if os.path.isdir(carpeta) and os.listdir(carpeta):
        z=shutil.make_archive('/content/'+nombre, 'zip', carpeta)
        print('Bajando', nombre+'.zip'); files.download(z)
    else:
        print('Sin archivos para', nombre)

---
### Notas
- **No se sube nada**: los AOIs ya están en la celda 4.
- `SOLO_1S = True` baja solo los productos con imagen. Poné `False` para incluir los `1M`.
- El access token se renueva solo si expira durante descargas largas.
- Para abrir las imágenes: **ESA SNAP 13+** (Microwave Toolbox, soporte BIOMASS beta).
